# Day 9 — K-Means Customer Clustering

This notebook evaluates K-Means cluster counts on the model-ready RFM feature matrix from Day 8, then fits the selected model. Cluster count is chosen from the observed validation metrics rather than assumed in advance.

**Selection methodology**
- Evaluate candidate `k` values from 2 through 8.
- Use **inertia** to assess within-cluster compactness; lower is better, but inertia always decreases as `k` increases.
- Use the **silhouette score** to assess separation and cohesion; higher is better.
- Inspect both metrics together and choose a defensible value, documenting the observed trade-off.
- Use a fixed `random_state=42` and `n_init=20` for reproducibility.
- Cluster labels are arbitrary identifiers; they are not business segment names.

No cluster count, metric, or segment result is hard-coded.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.eda import positive_sales_view
from src.rfm_analysis import build_rfm
from src.clustering_prep import prepare_clustering_features
from src.kmeans_clustering import evaluate_k_range, fit_kmeans

online_retail = fetch_ucirepo(id=352)
raw = online_retail.data.features.copy()
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['invoice_date'] = pd.to_datetime(raw['invoice_date'], errors='coerce')
for col in ['quantity', 'unit_price', 'customer_id']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
clean = raw.drop_duplicates().copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
sales = positive_sales_view(clean)
rfm = build_rfm(sales)
_, scaled_features, _ = prepare_clustering_features(rfm)
print(f'Customer rows available for clustering: {len(scaled_features):,}')

## 1. Evaluate candidate cluster counts

In [ ]:
evaluation = evaluate_k_range(scaled_features, range(2, 9), random_state=42, n_init=20)
evaluation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(evaluation['k'], evaluation['inertia'], marker='o')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('K-Means elbow curve')
ax.set_xticks(evaluation['k'])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(evaluation['k'], evaluation['silhouette_score'], marker='o')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.set_title('K-Means silhouette comparison')
ax.set_xticks(evaluation['k'])
plt.tight_layout()
plt.show()

## 2. Select k from observed evidence

The following cell reports the best silhouette candidate and the inertia values. Treat the highest silhouette score as evidence rather than an automatic business decision; if the curve suggests a nearby alternative with materially better compactness or interpretability, document that rationale before fitting the final model.

In [ ]:
best_silhouette_k = int(evaluation.loc[evaluation['silhouette_score'].idxmax(), 'k'])
print(f'Highest observed silhouette score occurs at k={best_silhouette_k}.')
evaluation.sort_values('silhouette_score', ascending=False).head(3)

## 3. Fit the selected model

For this reproducible workflow, the highest-silhouette candidate is used as the initial selected `k`. Day 10 will validate and interpret the resulting segments against business-facing RFM characteristics.

In [ ]:
selected_k = best_silhouette_k
model, labels = fit_kmeans(scaled_features, selected_k, random_state=42, n_init=20)
clustered_rfm = rfm.loc[scaled_features.index].copy()
clustered_rfm['cluster'] = labels
print(f'Selected k: {selected_k}')
clustered_rfm['cluster'].value_counts().sort_index()

In [ ]:
cluster_profile = (
    clustered_rfm.groupby('cluster')[['recency', 'frequency', 'monetary']]
    .agg(['count', 'mean', 'median'])
)
cluster_profile